In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


# ============================================================
# USER SETTINGS
# ============================================================

input_file = Path(
    r"C:\Users\mtpv1\Downloads\EIS-Data_Combined\All_EIS_results.csv"
)

# Output files/folders
output_file = input_file.parent / "EIS_results_consolidated.csv"
plot_folder = input_file.parent / "Plots"

plot_folder.mkdir(exist_ok=True)


# ============================================================
# LOAD RAW DATA
# ============================================================

df = pd.read_csv(input_file)


# ============================================================
# CIRCULAR STATISTICS FOR PHASE
# ============================================================

def circular_mean_deg(phases_deg):
    """
    Circular mean of phase angles in degrees.
    """
    phases = phases_deg.dropna()

    if len(phases) == 0:
        return np.nan

    phases_rad = np.deg2rad(phases)

    mean_sin = np.mean(np.sin(phases_rad))
    mean_cos = np.mean(np.cos(phases_rad))

    mean_angle_rad = np.arctan2(mean_sin, mean_cos)

    return np.rad2deg(mean_angle_rad)


def circular_std_deg(phases_deg):
    """
    Circular standard deviation in degrees.

    Returns NaN for n < 2 because variability cannot be
    estimated from a single sample.
    """
    phases = phases_deg.dropna()

    if len(phases) < 2:
        return np.nan

    phases_rad = np.deg2rad(phases)

    mean_sin = np.mean(np.sin(phases_rad))
    mean_cos = np.mean(np.cos(phases_rad))

    R = np.sqrt(mean_sin**2 + mean_cos**2)

    R = np.clip(R, 1e-15, 1.0)

    circular_std_rad = np.sqrt(-2 * np.log(R))

    return np.rad2deg(circular_std_rad)


# ============================================================
# CONSOLIDATE DATA
# ============================================================

consolidated = (
    df.groupby(
        ["Material", "Size", "Frequency_Hz"],
        as_index=False
    )
    .agg(
        # Number of measurements contributing to each point
        Sample_Count=("Impedance_Ohm", "count"),

        # Applied voltage
        Applied_Vrms_mV=("Applied_Vrms_mV", "mean"),
        Applied_Vrms_SD_mV=("Applied_Vrms_mV", "std"),

        # Cell voltage
        Cell_Vrms_mV=("Cell_Vrms_mV", "mean"),
        Cell_Vrms_SD_mV=("Cell_Vrms_mV", "std"),

        # Sense voltage
        Sense_Vrms_mV=("Sense_Vrms_mV", "mean"),
        Sense_Vrms_SD_mV=("Sense_Vrms_mV", "std"),

        # Current
        Current_nA=("Current_nA", "mean"),
        Current_SD_nA=("Current_nA", "std"),

        # Impedance
        Impedance_Ohm=("Impedance_Ohm", "mean"),
        Impedance_SD_Ohm=("Impedance_Ohm", "std"),

        # Phase
        Phase_deg=("Phase_deg", circular_mean_deg),
        Phase_SD_deg=("Phase_deg", circular_std_deg),

        # Fit/noise metrics
        Sense_SNR_dB=("Sense_SNR_dB", "mean"),
        Sense_SNR_SD_dB=("Sense_SNR_dB", "std"),

        Sense_R2=("Sense_R2", "mean"),
        Sense_R2_SD=("Sense_R2", "std"),

        Sense_Residual_mV=("Sense_Residual_mV", "mean"),
        Sense_Residual_SD_mV=("Sense_Residual_mV", "std"),
    )
)


# ============================================================
# SORT + ROUND
# ============================================================

consolidated = consolidated.sort_values(
    by=["Material", "Size", "Frequency_Hz"]
)

numeric_columns = consolidated.select_dtypes(
    include=[np.number]
).columns

consolidated[numeric_columns] = (
    consolidated[numeric_columns].round(6)
)


# ============================================================
# SAVE CONSOLIDATED CSV
# ============================================================

consolidated.to_csv(output_file, index=False)

print(f"Saved consolidated data to:\n{output_file}\n")


# ============================================================
# GENERATE PLOTS
# ============================================================

groups = consolidated.groupby(["Material", "Size"])

plot_count = 0


for (material, size), data in groups:

    data = data.sort_values("Frequency_Hz")

    # Convert identifiers to clean strings for filenames
    material_str = str(material)
    size_str = str(size)

    title = f"{material_str} – {size_str} µm"


    # ========================================================
    # IMPEDANCE VS FREQUENCY
    # ========================================================

    fig, ax = plt.subplots(figsize=(6, 4.5))

    ax.errorbar(
        data["Frequency_Hz"],
        data["Impedance_Ohm"],
        yerr=data["Impedance_SD_Ohm"],
        marker="o",
        linestyle="-",
        capsize=3,
        linewidth=1.5,
        markersize=5
    )

    ax.set_xscale("log")
    ax.set_yscale("log")

    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Impedance (Ω)")
    ax.set_title(title)

    ax.grid(
        True,
        which="both",
        alpha=0.25
    )

    fig.tight_layout()

    impedance_png = (
        plot_folder /
        f"{material_str}_{size_str}_Impedance.png"
    )

    impedance_pdf = (
        plot_folder /
        f"{material_str}_{size_str}_Impedance.pdf"
    )

    fig.savefig(
        impedance_png,
        dpi=300,
        bbox_inches="tight"
    )

    fig.savefig(
        impedance_pdf,
        bbox_inches="tight"
    )

    plt.close(fig)

    plot_count += 1


    # ========================================================
    # PHASE VS FREQUENCY
    # ========================================================

    fig, ax = plt.subplots(figsize=(6, 4.5))

    ax.errorbar(
        data["Frequency_Hz"],
        data["Phase_deg"],
        yerr=data["Phase_SD_deg"],
        marker="o",
        linestyle="-",
        capsize=3,
        linewidth=1.5,
        markersize=5
    )

    ax.set_xscale("log")

    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Phase (degrees)")
    ax.set_title(title)

    ax.grid(
        True,
        which="both",
        alpha=0.25
    )

    fig.tight_layout()

    phase_png = (
        plot_folder /
        f"{material_str}_{size_str}_Phase.png"
    )

    phase_pdf = (
        plot_folder /
        f"{material_str}_{size_str}_Phase.pdf"
    )

    fig.savefig(
        phase_png,
        dpi=300,
        bbox_inches="tight"
    )

    fig.savefig(
        phase_pdf,
        bbox_inches="tight"
    )

    plt.close(fig)

    plot_count += 1


# ============================================================
# FINISHED
# ============================================================

print(f"Generated {plot_count} plots.")
print(f"Plots saved to:\n{plot_folder}")

Saved consolidated data to:
C:\Users\mtpv1\Downloads\EIS-Data_Combined\EIS_results_consolidated.csv



c:\Users\mtpv1\Documents\Python\lib\site-packages\numpy\core\_methods.py:44: RuntimeWarning: invalid value encountered in reduce
  return umr_minimum(a, axis, None, out, keepdims, initial, where)
c:\Users\mtpv1\Documents\Python\lib\site-packages\numpy\core\_methods.py:40: RuntimeWarning: invalid value encountered in reduce
  return umr_maximum(a, axis, None, out, keepdims, initial, where)
c:\Users\mtpv1\Documents\Python\lib\site-packages\numpy\core\_methods.py:44: RuntimeWarning: invalid value encountered in reduce
  return umr_minimum(a, axis, None, out, keepdims, initial, where)
c:\Users\mtpv1\Documents\Python\lib\site-packages\numpy\core\_methods.py:40: RuntimeWarning: invalid value encountered in reduce
  return umr_maximum(a, axis, None, out, keepdims, initial, where)
c:\Users\mtpv1\Documents\Python\lib\site-packages\numpy\core\_methods.py:44: RuntimeWarning: invalid value encountered in reduce
  return umr_minimum(a, axis, None, out, keepdims, initial, where)
c:\Users\mtpv1\Docum

Generated 28 plots.
Plots saved to:
C:\Users\mtpv1\Downloads\EIS-Data_Combined\Plots
